# %% [markdown]
# ## Step 1: Load Raw Claims Data

In [12]:
# %%
import pandas as pd
import numpy as np
from datetime import timedelta

In [13]:
# Load raw claims
df_claims = pd.read_csv("raw_claims_data.csv", parse_dates=["Fill_Date"])

print(f"Raw Claims Loaded: {len(df_claims):,} rows")
print(f"Unique Patients  : {df_claims['Patient_ID'].nunique():,}")
print(f"\nColumns: {list(df_claims.columns)}")
print(f"\nSample records:")
print(df_claims.head(8).to_string())

Raw Claims Loaded: 2,184 rows
Unique Patients  : 500

Columns: ['Claim_ID', 'Patient_ID', 'Age', 'Age_Group', 'Gender', 'Region', 'Drug_Name', 'Therapy_Class', 'Prescriber_ID', 'Pharmacy_ID', 'Fill_Date', 'Days_Supply', 'Fill_Number']

Sample records:
    Claim_ID Patient_ID   Age Age_Group  Gender Region    Drug_Name    Therapy_Class Prescriber_ID Pharmacy_ID  Fill_Date  Days_Supply  Fill_Number
0  CLM000001    PAT0001  34.0     18-35  Female  Urban    Metformin  Type 2 Diabetes         DR015       PH005 2022-01-24         30.0          1.0
1  CLM000002    PAT0001  34.0     18-35  Female  Urban    Metformin  Type 2 Diabetes         DR015       PH005 2022-04-21         30.0          2.0
2  CLM000003    PAT0001  34.0     18-35  Female  Urban    Metformin  Type 2 Diabetes         DR015       PH005 2022-05-23         30.0          3.0
3  CLM000004    PAT0001  34.0     18-35  Female  Urban    Metformin  Type 2 Diabetes         DR015       PH005 2022-06-22         30.0          4.0
4  CLM00

C:\Users\DJ\AppData\Local\Temp\ipykernel_3304\695910917.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_claims = pd.read_csv("raw_claims_data.csv", parse_dates=["Fill_Date"])


# %% [markdown]
# ## Step 2: Define Measurement Period
#
# PDC and MPR are calculated over a fixed **measurement period** per patient.
# Standard pharma analytics uses a 12-month (365-day) observation window.
# The measurement period starts from the patient's **first prescription fill date**.


In [14]:
# %%
MEASUREMENT_DAYS = 365   # 12-month observation window
PDC_THRESHOLD    = 0.80  # Industry standard: PDC >= 0.80 = Adherent

# Calculate measurement window per patient
patient_periods = df_claims.groupby("Patient_ID").agg(
    Measurement_Start=("Fill_Date", "min"),
).reset_index()

patient_periods["Measurement_End"] = (
    patient_periods["Measurement_Start"] + timedelta(days=MEASUREMENT_DAYS)
)

print("Measurement periods assigned:")
print(patient_periods.head(5).to_string())

Measurement periods assigned:
  Patient_ID Measurement_Start Measurement_End
0    PAT0001        2022-01-24      2023-01-24
1    PAT0002        2022-01-10      2023-01-10
2    PAT0003        2022-01-07      2023-01-07
3    PAT0004        2022-01-05      2023-01-05
4    PAT0005        2022-01-29      2023-01-29


# %% [markdown]
# ## Step 3: Calculate NRx and TRx
#
# **NRx (New Prescription):** The patient's FIRST fill for a drug.
# Indicates a new therapy start — key metric for brand uptake tracking.
#
# **TRx (Total Prescriptions):** ALL fills including refills.
# Represents total prescription volume — core commercial metric.

In [15]:
# %%
# Sort claims by patient and fill date
df_claims = df_claims.sort_values(["Patient_ID", "Fill_Date"]).reset_index(drop=True)

# NRx = first fill per patient per drug
df_claims["Rx_Type"] = df_claims.groupby(
    ["Patient_ID", "Drug_Name"]
)["Fill_Date"].transform(lambda x: ["NRx"] + ["TRx"] * (len(x) - 1))

nrx_count = (df_claims["Rx_Type"] == "NRx").sum()
trx_count = (df_claims["Rx_Type"] == "TRx").sum()

print(f"NRx (New Prescriptions) : {nrx_count:,}  ({nrx_count/len(df_claims):.1%})")
print(f"TRx (Refills)           : {trx_count:,}  ({trx_count/len(df_claims):.1%})")
print(f"\nNRx/TRx by Drug:")
print(df_claims.groupby(["Drug_Name", "Rx_Type"]).size().unstack(fill_value=0).to_string())

NRx (New Prescriptions) : 500  (22.9%)
TRx (Refills)           : 1,626  (74.5%)

NRx/TRx by Drug:
Rx_Type      NRx  TRx
Drug_Name            
Glipizide    161  503
Metformin    226  864
Sitagliptin  113  259


# %% [markdown]
# ## Step 4: Calculate Refill Gap
#
# **Refill Gap** = Number of days between when a prescription SHOULD have been
# refilled (Fill_Date + Days_Supply) and when it WAS actually refilled.
#
# High refill gaps indicate patients are running out of medication before refilling —
# a strong early warning signal for non-adherence.

In [16]:
# %%
# Calculate expected next fill date and actual gap
df_claims["Expected_Next_Fill"] = df_claims["Fill_Date"] + pd.to_timedelta(
    df_claims["Days_Supply"], unit="D"
)

# Shift to get actual next fill date within same patient
df_claims["Actual_Next_Fill"] = df_claims.groupby("Patient_ID")["Fill_Date"].shift(-1)

# Refill gap = actual next fill - expected next fill (positive = late)
df_claims["Refill_Gap_Days"] = (
    df_claims["Actual_Next_Fill"] - df_claims["Expected_Next_Fill"]
).dt.days

# For last fill per patient, gap is NaN — fill with 0
df_claims["Refill_Gap_Days"] = df_claims["Refill_Gap_Days"].fillna(0).clip(lower=0).astype(int)

print("Refill Gap calculated:")
print(df_claims[["Patient_ID", "Drug_Name", "Fill_Date", "Days_Supply",
                  "Expected_Next_Fill", "Actual_Next_Fill",
                  "Refill_Gap_Days"]].head(10).to_string())

print(f"\nAvg Refill Gap overall : {df_claims['Refill_Gap_Days'].mean():.1f} days")
print(f"Max Refill Gap         : {df_claims['Refill_Gap_Days'].max()} days")

Refill Gap calculated:
  Patient_ID    Drug_Name  Fill_Date  Days_Supply Expected_Next_Fill Actual_Next_Fill  Refill_Gap_Days
0    PAT0001    Metformin 2022-01-24         30.0         2022-02-23       2022-04-21               57
1    PAT0001    Metformin 2022-04-21         30.0         2022-05-21       2022-05-23                2
2    PAT0001    Metformin 2022-05-23         30.0         2022-06-22       2022-06-22                0
3    PAT0001    Metformin 2022-06-22         30.0         2022-07-22       2022-08-01               10
4    PAT0001    Metformin 2022-08-01         90.0         2022-10-30       2022-10-30                0
5    PAT0001    Metformin 2022-10-30         30.0         2022-11-29       2022-11-29                0
6    PAT0001    Metformin 2022-11-29         30.0         2022-12-29              NaT                0
7    PAT0002  Sitagliptin 2022-01-10         90.0         2022-04-10       2022-08-15              127
8    PAT0002  Sitagliptin 2022-08-15         30.0 

# %% [markdown]
# ## Step 5: Calculate PDC (Proportion of Days Covered)
#
# **PDC Formula:**
# PDC = Total unique days covered by medication ÷ Total days in measurement period
#
# PDC accounts for overlapping fills correctly — if patient refills early,
# the extra days are NOT double counted. This is the AMCP standard definition.
#
# **PDC >= 0.80 = Adherent** (industry standard threshold)


In [17]:
# %%
def calculate_pdc(patient_claims, measurement_start, measurement_end):
    """
    Calculate PDC for a single patient using AMCP standard method.
    
    Steps:
    1. Create a coverage array for each day in measurement period
    2. For each fill, mark covered days (handling early refills correctly)
    3. PDC = covered days / total measurement days
    """
    total_days = (measurement_end - measurement_start).days
    covered_days = np.zeros(total_days, dtype=int)  # 0 = not covered, 1 = covered
    
    for _, fill in patient_claims.iterrows():
        fill_start = int((fill["Fill_Date"] - measurement_start).days)
        fill_end   = int(fill_start + fill["Days_Supply"])
        
        # Clip to measurement window
        fill_start = max(0, fill_start)
        fill_end   = min(total_days, fill_end)
        
        if fill_start < fill_end:
            covered_days[fill_start:fill_end] = 1
    
    pdc = covered_days.sum() / total_days
    return round(pdc, 4)

# Calculate PDC for all patients
print("Calculating PDC for all patients...")

pdc_results = []

for _, period in patient_periods.iterrows():
    patient_id  = period["Patient_ID"]
    meas_start  = period["Measurement_Start"]
    meas_end    = period["Measurement_End"]
    
    patient_fills = df_claims[df_claims["Patient_ID"] == patient_id].copy()
    
    # Only use fills within measurement period
    patient_fills = patient_fills[
        (patient_fills["Fill_Date"] >= meas_start) &
        (patient_fills["Fill_Date"] <= meas_end)
    ]
    
    if len(patient_fills) == 0:
        pdc = 0.0
    else:
        pdc = calculate_pdc(patient_fills, meas_start, meas_end)
    
    pdc_results.append({
        "Patient_ID": patient_id,
        "PDC_Score":  pdc
    })

df_pdc = pd.DataFrame(pdc_results)
print(f"✅ PDC calculated for {len(df_pdc):,} patients")
print(f"\nPDC Distribution:")
print(df_pdc["PDC_Score"].describe().round(3).to_string())

Calculating PDC for all patients...
✅ PDC calculated for 500 patients

PDC Distribution:
count    500.000
mean       0.569
std        0.239
min        0.082
25%        0.394
50%        0.575
75%        0.740
max        0.994


# %% [markdown]
# ## Step 6: Calculate MPR (Medication Possession Ratio)
#
# **MPR Formula:**
# MPR = Total days supply dispensed ÷ Total days in measurement period
#
# Unlike PDC, MPR DOES count early refills — can exceed 1.0 if patient stockpiles.
# MPR is simpler to calculate but less accurate than PDC for adherence measurement.
# Both metrics are used in pharma analytics — showing both demonstrates depth.


In [18]:
# %%
def calculate_mpr(patient_claims, measurement_days):
    """
    MPR = Sum of all days supply dispensed / measurement period length
    Capped at 1.0 (can't be more than 100% covered)
    """
    total_days_dispensed = patient_claims["Days_Supply"].sum()
    mpr = total_days_dispensed / measurement_days
    return round(min(mpr, 1.0), 4)

mpr_results = []

for _, period in patient_periods.iterrows():
    patient_id  = period["Patient_ID"]
    meas_start  = period["Measurement_Start"]
    meas_end    = period["Measurement_End"]
    
    patient_fills = df_claims[
        (df_claims["Patient_ID"] == patient_id) &
        (df_claims["Fill_Date"] >= meas_start) &
        (df_claims["Fill_Date"] <= meas_end)
    ]
    
    if len(patient_fills) == 0:
        mpr = 0.0
    else:
        mpr = calculate_mpr(patient_fills, MEASUREMENT_DAYS)
    
    mpr_results.append({
        "Patient_ID": patient_id,
        "MPR_Score":  mpr
    })

df_mpr = pd.DataFrame(mpr_results)
print(f"✅ MPR calculated for {len(df_mpr):,} patients")

✅ MPR calculated for 500 patients


# %% [markdown]
# ## Step 7: Build Patient-Level Summary Table
#
# Combine all calculated metrics into one patient-level table for EDA and ML.

In [19]:
# %%
# Get patient demographics from claims (one row per patient)
df_demographics = df_claims.groupby("Patient_ID").agg(
    Age          =("Age", "first"),
    Age_Group    =("Age_Group", "first"),
    Gender       =("Gender", "first"),
    Region       =("Region", "first"),
    Drug_Name    =("Drug_Name", "first"),
    Therapy_Class=("Therapy_Class", "first"),
    Total_Fills  =("Claim_ID", "count"),
    NRx_Count    =("Rx_Type", lambda x: (x == "NRx").sum()),
    TRx_Count    =("Rx_Type", lambda x: (x == "TRx").sum()),
    Avg_Refill_Gap=("Refill_Gap_Days", "mean"),
    Max_Refill_Gap=("Refill_Gap_Days", "max"),
    Total_Days_Supply=("Days_Supply", "sum"),
).reset_index()

# Merge PDC and MPR
df_patients = df_demographics.merge(df_pdc, on="Patient_ID")
df_patients = df_patients.merge(df_mpr, on="Patient_ID")

# Add adherence flag
df_patients["Adherent"] = df_patients["PDC_Score"].apply(
    lambda x: "Yes" if x >= PDC_THRESHOLD else "No"
)

# Add adherence label for ML
df_patients["Adherent_Flag"] = (df_patients["PDC_Score"] >= PDC_THRESHOLD).astype(int)

# Round numeric columns
df_patients["Avg_Refill_Gap"] = df_patients["Avg_Refill_Gap"].round(1)
df_patients["PDC_Score"]      = df_patients["PDC_Score"].round(4)
df_patients["MPR_Score"]      = df_patients["MPR_Score"].round(4)

print("✅ Patient Summary Table Built")
print(f"Shape: {df_patients.shape}")
print(f"\nColumns: {list(df_patients.columns)}")
print(f"\nSample:")
print(df_patients.head(5).to_string())

✅ Patient Summary Table Built
Shape: (500, 17)

Columns: ['Patient_ID', 'Age', 'Age_Group', 'Gender', 'Region', 'Drug_Name', 'Therapy_Class', 'Total_Fills', 'NRx_Count', 'TRx_Count', 'Avg_Refill_Gap', 'Max_Refill_Gap', 'Total_Days_Supply', 'PDC_Score', 'MPR_Score', 'Adherent', 'Adherent_Flag']

Sample:
  Patient_ID   Age Age_Group  Gender      Region    Drug_Name    Therapy_Class  Total_Fills  NRx_Count  TRx_Count  Avg_Refill_Gap  Max_Refill_Gap  Total_Days_Supply  PDC_Score  MPR_Score Adherent  Adherent_Flag
0    PAT0001  34.0     18-35  Female       Urban    Metformin  Type 2 Diabetes            7          1          6             9.9              57              270.0     0.7397     0.7397       No              0
1    PAT0002  83.0     76-90    Male       Urban  Sitagliptin  Type 2 Diabetes            3          1          2            42.3             127              210.0     0.5753     0.5753       No              0
2    PAT0003  61.0     61-75    Male  Semi-Urban    Metformin  

# %% [markdown]
# ## Step 8: Validate Story — Confirm Trends Are Present

In [20]:
print("=" * 50)
print("QUICK DASHBOARD CHECK")
print("=" * 50)

print(f"Total Patients     : {len(df_patients)}")
print(f"Adherence Rate     : {(df_patients['Adherent']=='Yes').mean()*100:.2f}%")
print(f"Avg PDC Score      : {df_patients['PDC_Score'].mean():.2f}")
print(f"Avg Refill Gap     : {df_patients[df_patients['Adherent']=='No']['Avg_Refill_Gap'].mean():.2f} days")

print("\nPDC by Age Group:")
print(df_patients.groupby("Age_Group")["PDC_Score"].mean().reindex(
    ["18-35","36-45","46-60","61-75","76-90"]).round(2).to_string())

print("\nPDC by Drug:")
print(df_patients.groupby("Drug_Name")["PDC_Score"].mean().round(2).sort_values().to_string())

print("\nRefill Gap by Region:")
print(df_patients.groupby("Region")["Avg_Refill_Gap"].mean().round(2).sort_values(ascending=False).to_string())

QUICK DASHBOARD CHECK
Total Patients     : 500
Adherence Rate     : 22.00%
Avg PDC Score      : 0.57
Avg Refill Gap     : 37.12 days

PDC by Age Group:
Age_Group
18-35    0.45
36-45    0.56
46-60    0.72
61-75    0.67
76-90    0.59

PDC by Drug:
Drug_Name
Sitagliptin    0.43
Glipizide      0.57
Metformin      0.64

Refill Gap by Region:
Region
Rural         42.23
Semi-Urban    29.92
Urban         22.51
